<a href="https://colab.research.google.com/github/byshadowoz/Algorithms_in_python/blob/main/Organizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WARNING!
This could download heavy files, i recommend run this in a cloud ambience such as Google colab

In [ ]:
!rm -r sample_data

In [ ]:
!wget -O acousticbrainz-0.tar.zst "https://data.metabrainz.org/pub/musicbrainz/acousticbrainz/dumps/acousticbrainz-lowlevel-json-20220623/acousticbrainz-lowlevel-json-20220623-29.tar.zst"

--2024-12-05 12:11:57--  https://data.metabrainz.org/pub/musicbrainz/acousticbrainz/dumps/acousticbrainz-lowlevel-json-20220623/acousticbrainz-lowlevel-json-20220623-29.tar.zst
Resolving data.metabrainz.org (data.metabrainz.org)... 138.201.203.43
Connecting to data.metabrainz.org (data.metabrainz.org)|138.201.203.43|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9712191391 (9.0G) [application/octet-stream]
Saving to: ‘acousticbrainz-0.tar.zst’

acousticbrainz-0.ta 100%[===================>]   9.04G  19.1MB/s    in 10m 58s 

2024-12-05 12:22:56 (14.1 MB/s) - ‘acousticbrainz-0.tar.zst’ saved [9712191391/9712191391]



In [ ]:
!apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 49 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (588 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 123632 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!zstd -d acousticbrainz-0.tar.zst

acousticbrainz-0.tar.zst: 28821032960 bytes 


In [ ]:
!rm acousticbrainz-0.tar.zst

In [ ]:
!tar -xvf acousticbrainz-0.tar

Streaming output truncated to the last 5000 lines.
acousticbrainz-lowlevel-json-20220623/lowlevel/38/7/38775a85-1f9b-4faf-9187-9de348a0f2fd-4.json
acousticbrainz-lowlevel-json-20220623/lowlevel/54/5/545bea58-6d7d-4acb-a809-7c5d813737f1-0.json
acousticbrainz-lowlevel-json-20220623/lowlevel/d8/9/d899e0b6-12da-4247-854d-c8f4ce52aadb-0.json
acousticbrainz-lowlevel-json-20220623/lowlevel/b6/b/b6b15a99-b391-4ad3-89d8-56801692553c-0.json
acousticbrainz-lowlevel-json-20220623/lowlevel/bc/2/bc28f143-9c66-464c-8b39-acb084fe57d1-17.json
acousticbrainz-lowlevel-json-20220623/lowlevel/72/7/7279bb19-bdfd-4573-941d-4ac924db25aa-0.json
acousticbrainz-lowlevel-json-20220623/lowlevel/85/5/85528246-44ac-42e3-9697-f7d66ef5045e-0.json
acousticbrainz-lowlevel-json-20220623/lowlevel/75/f/75ff95db-864c-433d-a54c-ad351461258c-2.json
acousticbrainz-lowlevel-json-20220623/lowlevel/22/d/22d49bd3-1160-4c85-afc3-24d8aa5588d7-0.json
acousticbrainz-lowlevel-json-20220623/lowlevel/45/a/45ab9ab1-1b23-4b00-8c1f-192c1299

In [ ]:
!rm acousticbrainz-0.tar

In [ ]:
import os
import concurrent.futures
from textwrap import indent
from pathlib import Path
import json

lowlevel_path = Path("/content/acousticbrainz-lowlevel-json-20220623/lowlevel")
output_dir = Path("/content/output")
output_dir.mkdir(exist_ok=True)

with open("template.json", "r") as target_file:
    template = json.load(target_file)

def fill_structure(template, data):
    if isinstance(template, dict):
        return {
            key: fill_structure(template[key], data.get(key, {}))
            for key in template
        }
    elif isinstance(template, list):
        return data if isinstance(data, list) else []
    else:
        return data if isinstance(data, (int, float, str, list, dict)) else None

def process_json_file(json_file):
    try:
        if json_file.stat().st_size == 0:
            return
        with open(str(json_file), "r") as source_json:
            source_data = json.load(source_json)

        organized_data = fill_structure(template, source_data)

        output_path = output_dir / json_file.name
        with open(str(output_path), "w") as output_file:
            json.dump(organized_data, output_file, indent=4)

        print(f"Processed: {json_file} has been reorganized in {output_path}")
        os.remove(json_file)

    except Exception as e:
        print(f"Error processing {json_file}: {e}")
        if 'source_data' in locals():
            print(source_data.keys())

json_files = [file for file in lowlevel_path.rglob("*.json") if file.stat().st_size > 0]

with concurrent.futures.ThreadPoolExecutor() as executor:
    executor.map(process_json_file, json_files)
    print("All files have been processed.")
    print(f"Total files processed: {len(json_files)}")
    print(f"Total files left: {len(list(lowlevel_path.rglob('*.json')))}")
    print(f"Total files in output: {len(list(output_dir.rglob('*.json')))}")


Streaming output truncated to the last 5000 lines.
Processed: /content/acousticbrainz-lowlevel-json-20220623/lowlevel/a6/e/a6e0e737-b1fd-45ec-89d5-9daf06c1d306-1.json has been reorganized in /content/output/a6e0e737-b1fd-45ec-89d5-9daf06c1d306-1.json
Processed: /content/acousticbrainz-lowlevel-json-20220623/lowlevel/a6/e/a6e82f6a-1faa-4245-b0ab-70a839eab8a9-84.json has been reorganized in /content/output/a6e82f6a-1faa-4245-b0ab-70a839eab8a9-84.json
Processed: /content/acousticbrainz-lowlevel-json-20220623/lowlevel/a6/e/a6e0a71a-e97a-4d14-be10-48e48ce1af37-4.json has been reorganized in /content/output/a6e0a71a-e97a-4d14-be10-48e48ce1af37-4.jsonProcessed: /content/acousticbrainz-lowlevel-json-20220623/lowlevel/a6/e/a6e87ce1-239e-468a-a116-58a4b2958138-8.json has been reorganized in /content/output/a6e87ce1-239e-468a-a116-58a4b2958138-8.json

Processed: /content/acousticbrainz-lowlevel-json-20220623/lowlevel/a6/e/a6e6ae08-1d20-4921-9a1a-dbcdba58aa7f-6.json has been reorganized in /conten

In [ ]:
!rm -r "/content/acousticbrainz-lowlevel-json-20220623"

In [ ]:
!zip -r "/content/data_organized.zip" "/content/output"

Streaming output truncated to the last 5000 lines.
  adding: content/output/a5452749-7710-44a6-b677-ebacf165f99a-24.json (deflated 76%)
  adding: content/output/9839e9aa-cc5c-4a74-987d-a84116a10b50-97.json (deflated 76%)
  adding: content/output/66a7bf6b-aa4a-406d-a633-4301708fc9d6-0.json (deflated 76%)
  adding: content/output/d413e767-4457-4279-9b14-b91cc9b99d0b-139.json (deflated 76%)
  adding: content/output/0a955960-2b74-497c-9f54-0cbd977fb8b8-20.json (deflated 76%)
  adding: content/output/e661d2cc-2499-4c5e-ae5c-e0e55b3cf137-25.json (deflated 76%)
  adding: content/output/ecda5101-4c49-4b08-bb98-8c5688a3a1e2-48.json (deflated 77%)
  adding: content/output/d1dafcdb-8503-43bf-b3b8-49f1a396fe22-24.json (deflated 76%)
  adding: content/output/7fca893d-ace2-4f4d-b522-e4a4d6149b69-0.json (deflated 76%)
  adding: content/output/995fa93f-d07f-4ee4-8655-ad28ecb5921e-0.json (deflated 76%)
  adding: content/output/65d51599-2ee4-47b6-a6c8-488aa2ea2c27-4.json (deflated 76%)
  adding: content

In [ ]:
!rm -r "/content/output"

In [1]:
import google.colab
google.colab.files.download("/content/data_organized.zip") # no necesary

FileNotFoundError: Cannot find file: /content/data_organized.zip